In [ ]:
import requests
import json
import os

# --- Configuration ---
# Obtain a free API key from virustotal.com
# Store it securely, e.g., in an environment variable or a secrets manager.
# For this lab, we'll assume it's in an environment variable.
VIRUSTOTAL_API_KEY = os.environ.get('VIRUSTOTAL_API_KEY')

if not VIRUSTOTAL_API_KEY:
    print("Error: VIRUSTOTAL_API_KEY environment variable not set.")
    print("Please set it before running this script.")
    # In a real SOAR, you might have a default or prompt for it.
    # For this lab, we'll use a placeholder and it will fail gracefully.
    VIRUSTOTAL_API_KEY = 'YOUR_PLACEHOLDER_API_KEY' # This will cause the API call to fail

VT_IP_REPORT_URL = 'https://www.virustotal.com/api/v3/ip_addresses/{ip}'

# --- AI-Enhanced Enrichment Logic ---
def enrich_ip_with_virustotal(ip_address):
    if VIRUSTOTAL_API_KEY == 'YOUR_PLACEHOLDER_API_KEY':
        print("Skipping VirusTotal lookup: API key not configured.")
        return None

    print(f"Querying VirusTotal for IP: {ip_address}...")
    headers = {
        'x-apikey': VIRUSTOTAL_API_KEY
    }
    url = VT_IP_REPORT_URL.format(ip=ip_address)

    try:
        response = requests.get(url, headers=headers, timeout=10) # Added timeout
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        
        report = response.json()
        
        # --- AI-driven analysis of the report (simplified) ---
        # In a real scenario, AI would parse unstructured data, identify TTPs, etc.
        # Here, we'll focus on key indicators.
        
        analysis_results = {
            'ip_address': ip_address,
            'malicious_votes': report['data']['attributes'].get('last_analysis_stats', {}).get('malicious', 0),
            'suspicious_votes': report['data']['attributes'].get('last_analysis_stats', {}).get('suspicious', 0),
            'harmless_votes': report['data']['attributes'].get('last_analysis_stats', {}).get('harmless', 0),
            'undetected_votes': report['data']['attributes'].get('last_analysis_stats', {}).get('undetected', 0),
            'country': report['data']['attributes'].get('country', 'N/A'),
            'as_owner': report['data']['attributes'].get('as_owner', 'N/A'),
            'tags': report['data']['attributes'].get('tags', [])
        }

        # Simple AI-like decision making based on votes
        risk_score = 0
        if analysis_results['malicious_votes'] > 5: # Threshold for high risk
            risk_score = 100
        elif analysis_results['malicious_votes'] > 2 or analysis_results['suspicious_votes'] > 5:
            risk_score = 75
        elif analysis_results['malicious_votes'] > 0:
            risk_score = 50
        else:
            risk_score = 25
        
        analysis_results['risk_score'] = risk_score

        print(f"VirusTotal lookup complete. Risk Score: {risk_score}")
        return analysis_results

    except requests.exceptions.RequestException as e:
        print(f"Error querying VirusTotal API: {e}")
        if response and response.status_code == 401: # Unauthorized
            print("Check your VIRUSTOTAL_API_KEY.")
        elif response and response.status_code == 404: # Not Found
            print(f"IP address {ip_address} not found in VirusTotal database.")
        return None
    except KeyError as e:
        print(f"Error parsing VirusTotal response: Missing key {e}. Response structure might have changed.")
        # print(f"Raw response: {json.dumps(report, indent=2)}") # Uncomment for debugging
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# --- SOAR Playbook Integration Example ---
def run_enrichment_playbook_step(alert_data):
    print("\n--- Running Threat Intelligence Enrichment Step ---")
    
    # Assume alert_data is a dictionary containing alert details
    # Example: alert_data = {'source_ip': '1.2.3.4', 'destination_ip': '192.168.1.10', ...}
    
    # Extract IoCs (simplified)
    potential_iocs = []
    if 'source_ip' in alert_data and alert_data['source_ip'] != '127.0.0.1': # Avoid localhost
        potential_iocs.append(alert_data['source_ip'])
    if 'destination_ip' in alert_data and alert_data['destination_ip'] != '127.0.0.1':
        potential_iocs.append(alert_data['destination_ip'])
    # Add logic for domains, file hashes etc.

    if not potential_iocs:
        print("No relevant IoCs found in alert data for enrichment.")
        return alert_data # Return original data if no IoCs

    enriched_data = alert_data.copy()
    enriched_data['threat_intelligence'] = {}

    for ioc in set(potential_iocs): # Use set to avoid duplicate lookups
        if '.' in ioc: # Basic check if it looks like an IP address
            vt_results = enrich_ip_with_virustotal(ioc)
            if vt_results:
                enriched_data['threat_intelligence'][ioc] = vt_results
                
                # AI-driven decision making based on enrichment results
                if vt_results['risk_score'] >= 75: # High risk based on VT score
                    print(f"High risk IoC detected: {ioc} (Risk Score: {vt_results['risk_score']})")
                    # Example: Trigger automated blocking action
                    # block_ip_at_firewall(ioc)
                    enriched_data['priority'] = 'Critical'
                    enriched_data['action_recommendation'] = f"Block {ioc} and investigate further."
                elif vt_results['risk_score'] >= 50:
                    print(f"Medium risk IoC detected: {ioc} (Risk Score: {vt_results['risk_score']})")
                    enriched_data['priority'] = 'High'
                    enriched_data['action_recommendation'] = f"Investigate {ioc} further."
            else:
                print(f"Could not enrich IoC: {ioc}")
        # Add logic here for other IoC types (domains, hashes)

    print("--- Enrichment Step Complete ---")
    return enriched_data

# --- Main Execution Logic ---
if __name__ == "__main__":
    # Simulate an alert that needs enrichment
    sample_alert = {
        'alert_id': 'ALERT-12345',
        'timestamp': '2023-10-27T10:00:00Z',
        'source_ip': '203.0.113.10', # Example suspicious IP
        'destination_ip': '192.168.1.50', # Internal server
        'protocol': 'TCP',
        'port': 443,
        'alert_description': 'Suspicious outbound connection detected.'
    }

    print(f"Original Alert Data: {sample_alert}")
    
    # Run the enrichment playbook step
    processed_alert = run_enrichment_playbook_step(sample_alert)

    print("\n--- Processed Alert Data ---")
    print(json.dumps(processed_alert, indent=2))

    print("\nIf you see this message → your lab is working perfectly!")